# Phase 3 — Submission 04: Model 2 Wide Mistral Long-Flow Decoder (AdaLoRA Adaptive Allocation)

**Task:** Rhetorical Role Prediction in Legal Documents (Corporate Law — `CL` & Information Technology — `IT`)  
**Task Type:** Sentence-Level Multiclass Classification (13 Official Classes)  
**Primary Evaluation Metric:** Macro-F1 Score  
**Target Backbone:** Phase 1 Model 2 Wide Mistral Long-Flow Decoder (136.0M, $L=12, d=896, H_q=14, H_{kv}=7$, GQA 2:1, SwiGLU $d_{ff}=2432$, RoPE)  
**Pretrained Hub Location:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `model2-wide-mistral-136m`)  
**Target Hub Destination:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `Phase_3/sub04_model2_adalora/`)  

---

### Architectural & Adaptation Highlights
1. **Wide 896-Dim Mistral Backbone:** Leverages Model 2's wide 896-dimensional latent representation ($L=12$, SwiGLU $d_{ff}=2432$) with 2:1 Grouped-Query Attention for high feature expressivity.
2. **AdaLoRA Adaptive Rank Allocation:** Employs Zhang et al. (2023) SVD-based adaptive rank pruning:
   * **Initial Rank ($r_{\text{init}}=24$):** Overparameterized initial adapter budget.
   * **Target Rank ($r_{\text{target}}=12$):** Iteratively pruned down based on importance scores $S_{ij} = |P_{ij} \nabla_{P_{ij}} \mathcal{L}|$.
   * **SVD Triplet Factorization:** $\Delta W = P \Lambda Q$ parameterized by `lora_B` ($P$), `lora_E` ($\Lambda$), and `lora_A` ($Q$).
3. **Orthogonal Regularization Loss:** Enforces approximate orthogonality on the singular matrix triplets:
   $$\mathcal{L}_{\text{orth}} = \sum_{\ell} \left( \| P_\ell^T P_\ell - I \|_F^2 + \| Q_\ell Q_\ell^T - I \|_F^2 \right)$$
   Combined loss: $\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{CE}} + \lambda_{\text{orth}} \mathcal{L}_{\text{orth}}$ with $\lambda_{\text{orth}} = 0.5$.
4. **Step-Calibrated Rank Allocation:** Accurately synchronizes `model.update_and_allocate(step)` on every unscaled optimizer step, pruning lower-importance singular values while safeguarding the optimizer state.
5. **512-Token Packed Context Pipeline (`pack_fixed`):** Context packing with 128-token overlap and earliest-window deduplication (`is_overlap=True` masked out during loss calculation).
6. **Multi-Modal Feature Fusion Head:** 2-Layer MLP classification head concatenating normalized sentence representations at EOS ($d=896$), 16-dim Fourier relative position embeddings ($\text{rel\_pos} = i / \max(1, N - 1)$), and 8-dim learned domain embeddings: $\text{Linear}(896 + 24 \to 256) \to \text{GELU} \to \text{Dropout}(0.1) \to \text{Linear}(256 \to 13)$.


In [ ]:
# Cell 2: Imports, Environment Setup & Upfront Hugging Face Authentication
import os
import sys
import json
import math
import time
import zipfile
import random
import getpass
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import PreTrainedTokenizerFast, AutoConfig, AutoModel, get_cosine_schedule_with_warmup
# Bypass Kaggle/Colab incompatible pre-installed torchao version check in PEFT
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

from peft import AdaLoraConfig, get_peft_model
from huggingface_hub import hf_hub_download, HfApi

# 1. Reproducible Seed Initialization
def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Compute Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

# 3. Repository and Target Configuration
MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
TARGET_SUBFOLDER = "Phase_3/sub04_model2_adalora"
DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
BACKBONE_SUBFOLDER = "model2-wide-mistral-136m"

# 4. Upfront Write-Permission Token Check (Fails Fast within 5 seconds if invalid)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF") or os.environ.get("HUGGINGFACE_TOKEN") or ""
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HF")
    except Exception:
        pass

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass
    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO, exist_ok=True, token=HF_TOKEN)
        print(f"Successfully verified write permissions on Hugging Face Hub: '{MODEL_REPO}'")
    except Exception as e:
        print(f"Warning: Upfront HF token verification encountered: {e}")
else:
    print("Notice: HF_TOKEN not found in environment. You can enter it before upload.")


In [ ]:
# Cell 3: Universal Dataset Loader for CL & IT Splits
LABEL2ID = {
    "Fact": 0, "Issue": 1, "ArgumentPetitioner": 2, "ArgumentRespondent": 3,
    "Statute": 4, "Dissent": 5, "PrecedentReliedUpon": 6, "PrecedentNotReliedUpon": 7,
    "PrecedentOverruled": 8, "RulingByLowerCourt": 9, "RatioOfTheDecision": 10,
    "RulingByPresentCourt": 11, "None": 12
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 13
DOMAIN2ID = {"CL": 0, "IT": 1}
ID2DOMAIN = {0: "CL", 1: "IT"}

@dataclass
class LegalDocument:
    doc_id: str
    domain: str
    domain_id: int
    sentences: List[str]
    labels: Optional[List[str]] = None
    label_ids: Optional[List[int]] = None

def normalize_dataset_records(raw_data: Any, domain: str, label2id: Dict[str, int] = LABEL2ID) -> List[LegalDocument]:
    domain_id = DOMAIN2ID.get(domain.upper(), 0)
    normalized: List[LegalDocument] = []
    if isinstance(raw_data, list):
        for item in raw_data:
            if not isinstance(item, dict):
                continue
            doc_id = str(item.get("id") or item.get("doc_id") or item.get("name") or "")
            sentences = item.get("text") or item.get("sentences") or []
            labels = item.get("labels") or item.get("label") or None
            label_ids = None
            if labels is not None:
                assert len(sentences) == len(labels), f"Length mismatch in doc {doc_id}"
                label_ids = []
                for lbl in labels:
                    if lbl not in label2id:
                        raise ValueError(f"CRITICAL: Unknown label '{lbl}' in doc '{doc_id}'!")
                    label_ids.append(label2id[lbl])
            normalized.append(LegalDocument(doc_id=doc_id, domain=domain.upper(), domain_id=domain_id, sentences=sentences, labels=labels, label_ids=label_ids))
    elif isinstance(raw_data, dict):
        for doc_id, val in raw_data.items():
            if isinstance(val, list):
                normalized.append(LegalDocument(doc_id=str(doc_id), domain=domain.upper(), domain_id=domain_id, sentences=val, labels=None, label_ids=None))
            elif isinstance(val, dict):
                sentences = val.get("text") or val.get("sentences") or []
                labels = val.get("labels") or val.get("label") or None
                label_ids = None
                if labels is not None:
                    label_ids = []
                    for lbl in labels:
                        if lbl not in label2id:
                            raise ValueError(f"CRITICAL: Unknown label '{lbl}' in doc '{doc_id}'!")
                        label_ids.append(label2id[lbl])
                normalized.append(LegalDocument(doc_id=str(doc_id), domain=domain.upper(), domain_id=domain_id, sentences=sentences, labels=labels, label_ids=label_ids))
    return normalized

def load_all_datasets(repo_id: str = DATASET_REPO):
    print("Downloading Phase 3 datasets from Hugging Face Hub...")
    vocab_path = hf_hub_download(repo_id, "Phase_3_RR/Train/label_vocab.json", repo_type="dataset", token=HF_TOKEN or None)
    with open(vocab_path, "r", encoding="utf-8") as f:
        raw_vocab = json.load(f)
    if isinstance(raw_vocab, list):
        vocab = {lbl: i for i, lbl in enumerate(raw_vocab)}
    else:
        vocab = raw_vocab
    assert vocab == LABEL2ID, f"Vocabulary parity mismatch! Hub: {vocab} vs Code: {LABEL2ID}"
        
    splits = {
        "cl_train": ("Phase_3_RR/Train/CL_train.json", "CL"),
        "it_train": ("Phase_3_RR/Train/IT_train.json", "IT"),
        "cl_dev":   ("Phase_3_RR/Train/CL_dev.json", "CL"),
        "it_dev":   ("Phase_3_RR/Train/IT_dev.json", "IT"),
        "cl_test":  ("Phase_3_RR/Test/CL_test.json", "CL"),
        "it_test":  ("Phase_3_RR/Test/IT_test.json", "IT"),
    }
    raw = {}
    for key, (path, _) in splits.items():
        loc = hf_hub_download(repo_id, path, repo_type="dataset", token=HF_TOKEN or None)
        with open(loc, "r", encoding="utf-8") as f:
            raw[key] = json.load(f)
            
    train_docs = normalize_dataset_records(raw["cl_train"], "CL") + normalize_dataset_records(raw["it_train"], "IT")
    dev_docs = normalize_dataset_records(raw["cl_dev"], "CL") + normalize_dataset_records(raw["it_dev"], "IT")
    test_docs = normalize_dataset_records(raw["cl_test"], "CL") + normalize_dataset_records(raw["it_test"], "IT")
    assert not ({d.doc_id for d in train_docs if d.domain == "CL"} & {d.doc_id for d in train_docs if d.domain == "IT"}), "Train doc ID collision!"
    assert not ({d.doc_id for d in dev_docs if d.domain == "CL"} & {d.doc_id for d in dev_docs if d.domain == "IT"}), "Dev doc ID collision!"
    assert not ({d.doc_id for d in test_docs if d.domain == "CL"} & {d.doc_id for d in test_docs if d.domain == "IT"}), "Test doc ID collision!"
    return train_docs, dev_docs, test_docs, vocab

train_docs, dev_docs, test_docs, label_vocab = load_all_datasets()
print(f"Loaded: {len(train_docs)} Train Docs (80 expected), {len(dev_docs)} Dev Docs (10 expected), {len(test_docs)} Test Docs (10 expected)")


In [ ]:
# Cell 4: Model 2 Tokenizer & 512-Token Context Packing (pack_fixed)
def load_model2_tokenizer(repo_id: str = MODEL_REPO):
    tok_path = hf_hub_download(repo_id, f"{BACKBONE_SUBFOLDER}/tokenizer.json", token=HF_TOKEN or None)
    tok = PreTrainedTokenizerFast(
        tokenizer_file=tok_path,
        bos_token="<bos>",
        eos_token="<eos>",
        pad_token="<pad>",
        unk_token="<unk>"
    )
    tok.padding_side = "right"
    assert tok.pad_token_id == 3 and tok.eos_token_id == 1, "Special token ID mismatch!"
    return tok

tokenizer = load_model2_tokenizer()

@dataclass
class SentenceInfo:
    doc_sent_idx: int
    rel_pos: float
    eos_pos: int
    label_id: int
    is_overlap: bool

@dataclass
class PackedWindow:
    doc_id: str
    domain: str
    domain_id: int
    window_idx: int
    input_ids: List[int]
    attention_mask: List[int]
    sentence_infos: List[SentenceInfo]

def pack_document_fixed(
    doc: LegalDocument,
    tokenizer: PreTrainedTokenizerFast,
    max_window_len: int = 512,
    max_overlap_len: int = 128,
    max_sent_len: int = 128,
) -> List[PackedWindow]:
    N = len(doc.sentences)
    if N == 0:
        return []

    # 1. Tokenize each sentence, truncate to max_sent_len - 1, and append native EOS token
    sent_tokens_list: List[List[int]] = []
    for s in doc.sentences:
        toks = tokenizer.encode(s, add_special_tokens=False)[: max_sent_len - 1]
        toks.append(tokenizer.eos_token_id)
        sent_tokens_list.append(toks)
    sent_lens = [len(toks) for toks in sent_tokens_list]

    packed_windows: List[PackedWindow] = []
    curr_idx = 0
    window_idx = 0

    while curr_idx < N:
        # Collect preceding overlap context (up to max_overlap_len)
        overlap_indices: List[int] = []
        overlap_tokens = 0
        if curr_idx > 0:
            for prev_i in range(curr_idx - 1, -1, -1):
                if overlap_tokens + sent_lens[prev_i] <= max_overlap_len:
                    overlap_indices.insert(0, prev_i)
                    overlap_tokens += sent_lens[prev_i]
                else:
                    break

        # Guaranteed progress check: trim overlap if adding next sentence exceeds 512 tokens
        while overlap_indices and (overlap_tokens + sent_lens[curr_idx] > max_window_len):
            removed = overlap_indices.pop(0)
            overlap_tokens -= sent_lens[removed]

        # Add new sentences starting from curr_idx
        new_indices: List[int] = []
        new_tokens = 0
        idx = curr_idx

        while idx < N:
            if overlap_tokens + new_tokens + sent_lens[idx] <= max_window_len:
                new_indices.append(idx)
                new_tokens += sent_lens[idx]
                idx += 1
            else:
                if not new_indices:
                    new_indices.append(idx)
                    new_tokens += sent_lens[idx]
                    idx += 1
                break

        assert len(new_indices) >= 1, f"Zero progress at doc {doc.doc_id}, sent {curr_idx}!"

        # Assemble packed sequence
        input_ids: List[int] = []
        sentence_infos: List[SentenceInfo] = []

        # (a) Overlap sentences (masked out during loss computation)
        for s_idx in overlap_indices:
            toks = sent_tokens_list[s_idx]
            input_ids.extend(toks)
            eos_pos = len(input_ids) - 1
            rel_pos = float(s_idx) / float(max(1, N - 1))
            sentence_infos.append(SentenceInfo(
                doc_sent_idx=s_idx,
                rel_pos=rel_pos,
                eos_pos=eos_pos,
                label_id=-100,  # Masked!
                is_overlap=True
            ))

        # (b) New sentences
        for s_idx in new_indices:
            toks = sent_tokens_list[s_idx]
            input_ids.extend(toks)
            eos_pos = len(input_ids) - 1
            rel_pos = float(s_idx) / float(max(1, N - 1))
            lbl_id = doc.label_ids[s_idx] if doc.label_ids is not None else -100
            sentence_infos.append(SentenceInfo(
                doc_sent_idx=s_idx,
                rel_pos=rel_pos,
                eos_pos=eos_pos,
                label_id=lbl_id,
                is_overlap=False
            ))

        attention_mask = [1] * len(input_ids)
        packed_windows.append(PackedWindow(
            doc_id=doc.doc_id,
            domain=doc.domain,
            domain_id=doc.domain_id,
            window_idx=window_idx,
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentence_infos=sentence_infos
        ))

        window_idx += 1
        curr_idx = idx

    return packed_windows

def pack_all_docs(docs: List[LegalDocument], tokenizer: PreTrainedTokenizerFast) -> List[PackedWindow]:
    windows = []
    for doc in docs:
        windows.extend(pack_document_fixed(doc, tokenizer))
    return windows

print("Packing context windows for train, dev, and test documents...")
train_windows = pack_all_docs(train_docs, tokenizer)
dev_windows = pack_all_docs(dev_docs, tokenizer)
test_windows = pack_all_docs(test_docs, tokenizer)

print(f"Context Windows Generated -> Train: {len(train_windows):,}, Dev: {len(dev_windows):,}, Test: {len(test_windows):,}")


In [ ]:
# Cell 5: PyTorch Dataset & Collation with Right-Side Padding
class PackedRRDataset(Dataset):
    def __init__(self, windows: List[PackedWindow]):
        self.windows = windows
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        return self.windows[idx]

def collate_packed_windows(batch: List[PackedWindow], pad_token_id: int = 3):
    batch_size = len(batch)
    lengths = [len(w.input_ids) for w in batch]
    max_len = max(lengths)

    padded_input_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long)
    padded_attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)

    b_indices: List[int] = []
    eos_positions: List[int] = []
    rel_positions: List[float] = []
    domain_ids_list: List[int] = []
    label_ids_list: List[int] = []
    is_overlap_list: List[bool] = []
    doc_ids_list: List[str] = []
    sent_idx_list: List[int] = []

    for b_idx, w in enumerate(batch):
        seq_len = len(w.input_ids)
        padded_input_ids[b_idx, :seq_len] = torch.tensor(w.input_ids, dtype=torch.long)
        padded_attention_mask[b_idx, :seq_len] = 1

        for s_info in w.sentence_infos:
            b_indices.append(b_idx)
            eos_positions.append(s_info.eos_pos)
            rel_positions.append(s_info.rel_pos)
            domain_ids_list.append(w.domain_id)
            label_ids_list.append(s_info.label_id)
            is_overlap_list.append(s_info.is_overlap)
            doc_ids_list.append(w.doc_id)
            sent_idx_list.append(s_info.doc_sent_idx)

    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "batch_idx": torch.tensor(b_indices, dtype=torch.long),
        "eos_pos": torch.tensor(eos_positions, dtype=torch.long),
        "rel_pos": torch.tensor(rel_positions, dtype=torch.float32),
        "domain_ids": torch.tensor(domain_ids_list, dtype=torch.long),
        "label_ids": torch.tensor(label_ids_list, dtype=torch.long),
        "is_overlap": torch.tensor(is_overlap_list, dtype=torch.bool),
        "doc_ids": doc_ids_list,
        "doc_sent_indices": sent_idx_list,
    }


In [ ]:
# Cell 6: Fourier Relative Position Encodings & 2-Layer MLP Head
class FourierPositionEncoding(nn.Module):
    """
    16-dimensional Fourier relative position encoding:
      rel_pos = i / max(1, N - 1) in [0.0, 1.0].
    Computes 8 frequencies: omega_k = pi * 2^k for k in 0..7.
    Outputs: [sin(rel_pos * omega), cos(rel_pos * omega)] -> 16 dimensions.
    """
    def __init__(self, dim: int = 16):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim
        self.num_bands = dim // 2
        freq_bands = (2.0 ** torch.arange(self.num_bands).float()) * math.pi
        self.register_buffer("freq_bands", freq_bands)
        
    def forward(self, rel_pos: torch.Tensor) -> torch.Tensor:
        scaled = rel_pos.unsqueeze(-1) * self.freq_bands.to(rel_pos.device, dtype=rel_pos.dtype)
        return torch.cat([torch.sin(scaled), torch.cos(scaled)], dim=-1)

class RhetoricalRoleMLPHead(nn.Module):
    """
    2-Layer MLP Classification Head:
      LayerNorm(d_model) -> Concat [LayerNorm(h_snt), 16-dim Fourier, 8-dim Domain]
      Linear(896 + 24 -> 256) -> GELU -> Dropout(0.1) -> Linear(256 -> 13).
    """
    def __init__(
        self,
        d_model: int = 896,
        num_classes: int = 13,
        fourier_dim: int = 16,
        domain_dim: int = 8,
        hidden_dim: int = 256,
        dropout: float = 0.1
    ):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.fourier_pos = FourierPositionEncoding(dim=fourier_dim)
        self.domain_embed = nn.Embedding(num_embeddings=2, embedding_dim=domain_dim)
        
        in_features = d_model + fourier_dim + domain_dim  # 896 + 16 + 8 = 920
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        
        self._init_weights()
        
    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
        nn.init.normal_(self.domain_embed.weight, mean=0.0, std=0.02)
        
    def forward(self, sentence_reprs: torch.Tensor, rel_pos: torch.Tensor, domain_ids: torch.Tensor) -> torch.Tensor:
        h_norm = self.layer_norm(sentence_reprs)
        fourier_feats = self.fourier_pos(rel_pos)
        domain_feats = self.domain_embed(domain_ids)
        combined = torch.cat([h_norm, fourier_feats, domain_feats], dim=-1)
        hidden = self.dropout(self.act(self.fc1(combined)))
        return self.fc2(hidden)


In [ ]:
# Cell 7: Model 2 Backbone Loading, AdaLoRA Setup & Orthogonality Loss
from safetensors.torch import load_file as safetensors_load_file

print(f"Downloading Model 2 configuration and weights from '{MODEL_REPO}/{BACKBONE_SUBFOLDER}'...")
cfg_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/config.json")
weights_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/model.safetensors")

config = AutoConfig.from_pretrained(cfg_path)
config.use_cache = False
config._name_or_path = MODEL_REPO
base_model = AutoModel.from_config(config)
base_model.config._name_or_path = MODEL_REPO

raw_state_dict = safetensors_load_file(weights_path)
cleaned_sd = {}
for k, v in raw_state_dict.items():
    if k.startswith("model."):
        cleaned_sd[k[len("model."):]] = v
    elif not k.startswith("lm_head."):
        cleaned_sd[k] = v

load_res = base_model.load_state_dict(cleaned_sd, strict=True)
assert len(load_res.missing_keys) == 0, f"Missing keys when loading backbone: {load_res.missing_keys}"
assert len(load_res.unexpected_keys) == 0, f"Unexpected keys when loading backbone: {load_res.unexpected_keys}"
print(f"Base model loaded successfully with strict=True! 100% of Phase 1 weights matched ({len(cleaned_sd)} tensors).")
print("Base Model 2 Wide Mistral Long-Flow Decoder loaded successfully!")

# Calculate total optimization steps for AdaLoRA budget scheduling
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
EPOCHS = 5

train_loader = DataLoader(
    PackedRRDataset(train_windows),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_packed_windows,
    drop_last=False
)

dev_loader = DataLoader(
    PackedRRDataset(dev_windows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_packed_windows
)

total_optimization_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS

# AdaLoRA SVD-triplet configuration
# Initial rank init_r=24 pruning down to target_r=12
adalora_config = AdaLoraConfig(
    init_r=24,
    target_r=12,
    beta1=0.85,
    beta2=0.85,
    tinit=int(0.15 * total_optimization_steps),
    tfinal=int(0.20 * total_optimization_steps),
    deltaT=10,
    lora_alpha=32.0,
    orth_reg_weight=0.5,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    total_step=total_optimization_steps,
    bias="none",
    task_type=None
)

model = get_peft_model(base_model, adalora_config)
head = RhetoricalRoleMLPHead(d_model=896, num_classes=13, dropout=0.1)

# CRITICAL: Move entire model and head to device post-injection to synchronize parameter devices
model = model.to(device)
head = head.to(device)

# Orthogonality Loss Computation for SVD Triplet Matrices (P @ Lambda @ Q)
def compute_orthogonality_loss(model: nn.Module) -> torch.Tensor:
    """
    Computes Frobenius norm regularization enforcing orthogonality on P (lora_B) and Q (lora_A):
      L_orth = sum ( ||P^T P - I||_F^2 + ||Q Q^T - I||_F^2 )
    """
    orth_loss = torch.tensor(0.0, device=device)
    count = 0
    for name, module in model.named_modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            A = module.lora_A["default"] if isinstance(module.lora_A, (nn.ParameterDict, nn.ModuleDict)) else module.lora_A
            B = module.lora_B["default"] if isinstance(module.lora_B, (nn.ParameterDict, nn.ModuleDict)) else module.lora_B
            if hasattr(A, "weight"): A = A.weight
            if hasattr(B, "weight"): B = B.weight
            
            r = A.shape[0]
            I_r = torch.eye(r, device=A.device, dtype=A.dtype)
            
            diff_B = torch.matmul(B.t(), B) - I_r
            diff_A = torch.matmul(A, A.t()) - I_r
            
            orth_loss = orth_loss + torch.norm(diff_B, p="fro") + torch.norm(diff_A, p="fro")
            count += 1
    return orth_loss / max(1, count)

# Audit Parameter Breakdown
trainable_adalora = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_head = sum(p.numel() for p in head.parameters() if p.requires_grad)
total_backbone = sum(p.numel() for p in model.parameters())
total_trainable = trainable_adalora + trainable_head

print("\n" + "="*55)
print(f"Model 2 Total Backbone Parameters:   {total_backbone:,}")
print(f"AdaLoRA Trainable Parameters (init): {trainable_adalora:,} ({trainable_adalora / total_backbone * 100:.3f}%)")
print(f"MLP Head Trainable Parameters:       {trainable_head:,}")
print(f"Total Trainable System Parameters:    {total_trainable:,} ({total_trainable / total_backbone * 100:.3f}%)")
print("="*55 + "\n")


In [ ]:
# Cell 8: Smoothed Inverse Square-Root Class Weighting
def compute_smoothed_weights(train_docs: List[LegalDocument], num_classes: int = 13, clamp_min: float = 0.35, clamp_max: float = 6.0):
    counts = [0] * num_classes
    for doc in train_docs:
        if doc.label_ids is not None:
            for lid in doc.label_ids:
                if 0 <= lid < num_classes:
                    counts[lid] += 1
    total = sum(counts)
    weights = []
    for c in range(num_classes):
        raw_w = math.sqrt(total / (num_classes * max(1, counts[c])))
        clamped_w = max(clamp_min, min(clamp_max, raw_w))
        weights.append(clamped_w)
    return torch.tensor(weights, dtype=torch.float32)

class_weights = compute_smoothed_weights(train_docs).to(device)
print("Smoothed Class Weights computed across all training instances:")
for lbl, idx in sorted(LABEL2ID.items(), key=lambda x: x[1]):
    print(f"  [{idx:2d}] {lbl:<24}: {class_weights[idx].item():.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)


In [ ]:
# Cell 9: Differential Optimization & Training Loop with AdaLoRA Rank Allocation
LR_BACKBONE = 2.5e-4
LR_HEAD = 1.0e-3
ORTH_REG_WEIGHT = 0.5

train_loader = DataLoader(
    PackedRRDataset(train_windows),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_packed_windows,
    drop_last=False
)

dev_loader = DataLoader(
    PackedRRDataset(dev_windows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_packed_windows
)

param_groups = [
    {
        "params": [p for p in model.parameters() if p.requires_grad],
        "lr": LR_BACKBONE,
        "weight_decay": 0.01,
    },
    {
        "params": [p for p in head.parameters() if p.requires_grad],
        "lr": LR_HEAD,
        "weight_decay": 0.01,
    },
]

optimizer = torch.optim.AdamW(param_groups)
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.10 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print(f"Optimizer: AdamW | AdaLoRA LR: {LR_BACKBONE} | Head LR: {LR_HEAD} | Steps: {total_steps}")

# Macro-F1 Metric Evaluation (Earliest-Window Deduplicated)
def compute_macro_f1(gold_labels: List[int], pred_labels: List[int], num_classes: int = 13):
    tp = [0] * num_classes
    fp = [0] * num_classes
    fn = [0] * num_classes
    for g, p in zip(gold_labels, pred_labels):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    f1s = []
    for c in range(num_classes):
        p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
        r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        f1s.append(f1)
    return sum(f1s) / num_classes

def evaluate_dev(model, head, data_loader, criterion, device):
    model.eval()
    head.eval()
    total_loss = 0.0
    valid_sentences = 0
    all_preds = []
    all_golds = []
    dev_logits_dict = {}
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            batch_idx = batch["batch_idx"].to(device)
            eos_pos = batch["eos_pos"].to(device)
            rel_pos = batch["rel_pos"].to(device)
            domain_ids = batch["domain_ids"].to(device)
            label_ids = batch["label_ids"].to(device)
            is_overlap = batch["is_overlap"].to(device)
            doc_ids = batch["doc_ids"]
            sent_indices = batch["doc_sent_indices"]
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            sent_reprs = outputs.last_hidden_state[batch_idx, eos_pos]
            
            logits = head(sent_reprs, rel_pos, domain_ids)
            
            targets = label_ids.clone()
            targets[is_overlap] = -100
            
            loss = criterion(logits, targets)
            num_new = (~is_overlap).sum().item()
            total_loss += loss.item() * num_new
            valid_sentences += num_new
            
            non_overlap_mask = (~is_overlap).cpu()
            preds = logits.argmax(dim=-1).cpu()[non_overlap_mask].tolist()
            golds = label_ids.cpu()[non_overlap_mask].tolist()
            
            all_preds.extend(preds)
            all_golds.extend(golds)

            logits_list = logits.cpu().tolist()
            overlap_list = is_overlap.cpu().tolist()
            for d_id, s_idx, l_row, ov in zip(doc_ids, sent_indices, logits_list, overlap_list):
                if not ov:
                    if d_id not in dev_logits_dict:
                        dev_logits_dict[d_id] = {}
                    dev_logits_dict[d_id][s_idx] = l_row
            
    avg_loss = total_loss / max(1, valid_sentences)
    macro_f1 = compute_macro_f1(all_golds, all_preds)
    return avg_loss, macro_f1, all_golds, all_preds, dev_logits_dict

# 5-Epoch Training Loop with AdaLoRA Budget Allocation
os.makedirs("checkpoint_sub04", exist_ok=True)
best_dev_f1 = 0.0
history = {"train_loss": [], "dev_loss": [], "dev_f1": []}

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
global_step = 0

print("Commencing 5-epoch training loop for Sub 04 (AdaLoRA Model 2 Wide Mistral)...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    head.train()
    running_loss = 0.0
    optimizer.zero_grad()
    
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_idx = batch["batch_idx"].to(device)
        eos_pos = batch["eos_pos"].to(device)
        rel_pos = batch["rel_pos"].to(device)
        domain_ids = batch["domain_ids"].to(device)
        label_ids = batch["label_ids"].to(device)
        is_overlap = batch["is_overlap"].to(device)
        
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            sent_reprs = outputs.last_hidden_state[batch_idx, eos_pos]
            
            logits = head(sent_reprs, rel_pos, domain_ids)
            
            targets = label_ids.clone()
            targets[is_overlap] = -100
            
            ce_loss = criterion(logits, targets)
            orth_loss = compute_orthogonality_loss(model)
            combined_loss = ce_loss + ORTH_REG_WEIGHT * orth_loss
            loss = combined_loss / GRAD_ACCUM_STEPS
            
        scaler.scale(loss).backward()
        running_loss += combined_loss.item()
        
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()
            
            # Step calibration: update and allocate AdaLoRA budget only on valid optimizer steps
            if scale_after >= scale_before:
                global_step += 1
                model.update_and_allocate(global_step)
                
            optimizer.zero_grad()
            scheduler.step()
            
    epoch_train_loss = running_loss / len(train_loader)
    dev_loss, dev_f1, _, _, _ = evaluate_dev(model, head, dev_loader, criterion, device)
    
    history["train_loss"].append(epoch_train_loss)
    history["dev_loss"].append(dev_loss)
    history["dev_f1"].append(dev_f1)
    
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} | Dev Loss: {dev_loss:.4f} | Dev Macro-F1: {dev_f1:.4f} | Steps: {global_step}")
    
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_adapter_weights = {k: v.cpu().clone() for k, v in model.state_dict().items() if any(pat in k for pat in ["lora_", "ia3_"])}
        torch.save(best_adapter_weights, "checkpoint_sub04/best_adapter_weights.pt")
        torch.save(head.state_dict(), "checkpoint_sub04/best_head_weights.pt")
        print(f" -> New Best Dev Macro-F1: {best_dev_f1:.4f}! Best Adapter and Head saved.")

total_train_time = time.time() - start_time
print(f"\nTraining completed in {total_train_time / 60:.2f} minutes. Best Dev Macro-F1: {best_dev_f1:.4f}")


In [ ]:
# Cell 10: Final Evaluation, Detailed Classification Report & Visualization
import matplotlib.pyplot as plt

# Load best checkpoint weights (Both Adapter AND Head)
best_adapter = torch.load("checkpoint_sub04/best_adapter_weights.pt", map_location=device)
model.load_state_dict(best_adapter, strict=False)
head.load_state_dict(torch.load("checkpoint_sub04/best_head_weights.pt", map_location=device))
print("Loaded best adapter AND best head weights for final evaluation.")

final_dev_loss, final_dev_f1, all_golds, all_preds, dev_logits_dict = evaluate_dev(model, head, dev_loader, criterion, device)

# Per-Class Performance Summary
tp = [0] * NUM_CLASSES
fp = [0] * NUM_CLASSES
fn = [0] * NUM_CLASSES
for g, p in zip(all_golds, all_preds):
    if g == p:
        tp[g] += 1
    else:
        fp[p] += 1
        fn[g] += 1

per_class_summary = {}
print("\n" + "="*70)
print(f"{'Class Name':<25} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<8}")
print("="*70)
for c in range(NUM_CLASSES):
    c_name = ID2LABEL[c]
    p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
    r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    support = tp[c] + fn[c]
    per_class_summary[c_name] = {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4), "support": support}
    print(f"{c_name:<25} {p:<10.4f} {r:<10.4f} {f1:<10.4f} {support:<8d}")
print("="*70)
print(f"{'Overall Macro-F1':<25} {'':<10} {'':<10} {final_dev_f1:<10.4f} {len(all_golds):<8d}")
print("="*70 + "\n")

# Export ordered dev_logits.pt
ordered_dev_logits = {}
for doc in dev_docs:
    d_map = dev_logits_dict[doc.doc_id]
    ordered_dev_logits[doc.doc_id] = torch.tensor([d_map[i] for i in range(len(doc.sentences))], dtype=torch.float32)
torch.save(ordered_dev_logits, "checkpoint_sub04/dev_logits.pt")
print(f"Exported dev soft logits for {len(ordered_dev_logits)} documents to checkpoint_sub04/dev_logits.pt")

# Save Metrics JSON
metrics_report = {
    "model_id": "model2-wide-mistral-136m",
    "adaptation": "AdaLoRA Adaptive Budget Allocation (init_r=24, target_r=12)",
    "best_dev_macro_f1": round(final_dev_f1, 4),
    "dev_loss": round(final_dev_loss, 4),
    "total_params": total_backbone + trainable_head,
    "trainable_params": total_trainable,
    "trainable_percentage": round((total_trainable / total_backbone) * 100, 4),
    "per_class": per_class_summary,
    "training_time_minutes": round(total_train_time / 60, 2),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
    "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD
}

with open("checkpoint_sub04/val_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_report, f, indent=2)

with open("checkpoint_sub04/training_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

# Plot Loss & Macro-F1 Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Combined Loss", marker="o")
plt.plot(range(1, EPOCHS + 1), history["dev_loss"], label="Dev Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sub 04: AdaLoRA Loss Curve")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history["dev_f1"], label="Dev Macro-F1", marker="^", color="green")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Sub 04: AdaLoRA Dev Macro-F1 Progression")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("checkpoint_sub04/loss_plot.png", dpi=150)
plt.show()
print("Saved evaluation report and loss plot to 'checkpoint_sub04/loss_plot.png'.")


In [ ]:
# Cell 11: Test Inference & Codabench Packaging
test_loader = DataLoader(
    PackedRRDataset(test_windows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_packed_windows
)

model.eval()
head.eval()

predictions_dict: Dict[str, Dict[int, str]] = {}
test_logits_dict: Dict[str, Dict[int, List[float]]] = {}

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_idx = batch["batch_idx"].to(device)
        eos_pos = batch["eos_pos"].to(device)
        rel_pos = batch["rel_pos"].to(device)
        domain_ids = batch["domain_ids"].to(device)
        is_overlap = batch["is_overlap"].cpu().tolist()
        doc_ids = batch["doc_ids"]
        sent_indices = batch["doc_sent_indices"]
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        sent_reprs = outputs.last_hidden_state[batch_idx, eos_pos]
        logits = head(sent_reprs, rel_pos, domain_ids)
        preds = logits.argmax(dim=-1).cpu().tolist()
        
        # Deduplication: Record prediction and logits ONLY from earliest window (is_overlap == False)
        logits_rows = logits.cpu().tolist()
        for d_id, s_idx, pred_id, l_row, overlap in zip(doc_ids, sent_indices, preds, logits_rows, is_overlap):
            if not overlap:
                if d_id not in predictions_dict:
                    predictions_dict[d_id] = {}
                    test_logits_dict[d_id] = {}
                predictions_dict[d_id][s_idx] = ID2LABEL[pred_id]
                test_logits_dict[d_id][s_idx] = l_row

# Convert dictionary of sentence maps into ordered lists
formatted_predictions: Dict[str, List[str]] = {}
for doc in test_docs:
    assert doc.doc_id in predictions_dict, f"Missing test document {doc.doc_id}!"
    doc_map = predictions_dict[doc.doc_id]
    sorted_sents = [doc_map[i] for i in range(len(doc.sentences))]
    formatted_predictions[doc.doc_id] = sorted_sents

# Assertions for Codabench Compliance
assert len(formatted_predictions) == len(test_docs), f"Doc count mismatch: {len(formatted_predictions)} != {len(test_docs)}"
for doc in test_docs:
    preds = formatted_predictions[doc.doc_id]
    assert len(preds) == len(doc.sentences), f"Sentence count mismatch in {doc.doc_id}"
    assert None not in preds, f"Missing prediction in {doc.doc_id}"
    for lbl in preds:
        assert lbl in LABEL2ID, f"Invalid label '{lbl}'"

print(f"Successfully generated test predictions for {len(formatted_predictions)} documents ({sum(len(v) for v in formatted_predictions.values()):,} sentences).")

# Export ordered test_logits.pt
ordered_test_logits = {}
for doc in test_docs:
    d_map = test_logits_dict[doc.doc_id]
    ordered_test_logits[doc.doc_id] = torch.tensor([d_map[i] for i in range(len(doc.sentences))], dtype=torch.float32)
torch.save(ordered_test_logits, "checkpoint_sub04/test_logits.pt")
print(f"Exported test soft logits to checkpoint_sub04/test_logits.pt ({os.path.getsize('checkpoint_sub04/test_logits.pt'):,} bytes).")

# Save prediction.json
pred_path = "checkpoint_sub04/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(formatted_predictions, f, indent=2)

# Package into submission.zip with prediction.json strictly at root
zip_path = "checkpoint_sub04/submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

# Verify submission.zip root integrity
with zipfile.ZipFile(zip_path, "r") as zf:
    infolist = zf.infolist()
    assert len(infolist) == 1, f"Expected exactly 1 file in submission.zip, found {len(infolist)}"
    assert infolist[0].filename == "prediction.json", f"Found {infolist[0].filename} instead of prediction.json"

print(f"Verified submission.zip: {os.path.getsize(zip_path):,} bytes, root file 'prediction.json'.")


In [ ]:
# Cell 12: Local Artifact Preparation & Atomic Hugging Face Hub Upload
output_dir = "checkpoint_sub04"

# 1. Save config and tokenizer
tokenizer.save_pretrained(output_dir)
config.save_pretrained(output_dir)

# 2. Save PEFT AdaLoRA adapter weights and custom head weights
model.save_pretrained(output_dir)
torch.save(head.state_dict(), os.path.join(output_dir, "mlp_head.pt"))

# Sanitize auto-generated README.md and adapter_config.json to pass Hugging Face YAML validation
readme_path = os.path.join(output_dir, "README.md")
readme_content = f"""---
base_model: {MODEL_REPO}
tags:
- legal
- legal-nlp
- rhetorical-roles
library_name: peft
---

# Phase 3 Sub 04: Model 2 Wide Mistral AdaLoRA
- **Base Model:** `{MODEL_REPO}` (subfolder: `{BACKBONE_SUBFOLDER}`)
- **Task:** Sentence-Level Rhetorical Role Prediction (13 Classes)
- **Dev Macro-F1:** {final_dev_f1:.4f}
"""
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

adapter_cfg_path = os.path.join(output_dir, "adapter_config.json")
if os.path.exists(adapter_cfg_path):
    with open(adapter_cfg_path, "r", encoding="utf-8") as f:
        acfg = json.load(f)
    acfg["base_model_name_or_path"] = MODEL_REPO
    with open(adapter_cfg_path, "w", encoding="utf-8") as f:
        json.dump(acfg, f, indent=2)

# 3. Verify all required artifacts exist locally
required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "mlp_head.pt",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "val_metrics.json",
    "training_evaluation_report.json",
    "loss_plot.png",
    "dev_logits.pt",
    "test_logits.pt",
    "prediction.json",
    "submission.zip"
]

# Note: adapter_model can be .safetensors or .bin depending on PEFT environment
missing_files = []
for f in required_files:
    if f == "adapter_model.safetensors" and not os.path.exists(os.path.join(output_dir, f)):
        if not os.path.exists(os.path.join(output_dir, "adapter_model.bin")):
            missing_files.append(f)
    elif not os.path.exists(os.path.join(output_dir, f)):
        missing_files.append(f)

assert not missing_files, f"Cannot proceed with upload! Missing local files: {missing_files}"
print(f"All required local artifacts verified present in '{output_dir}'!")

# 4. Atomic single-commit folder upload to Hugging Face Hub
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter Hugging Face Write Token: ")

api = HfApi(token=HF_TOKEN)
print(f"Initiating atomic upload to '{MODEL_REPO}' under '{TARGET_SUBFOLDER}'...")
api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message=f"Upload Phase 3 Sub 04 (AdaLoRA Model 2 Wide Mistral) - Dev F1: {final_dev_f1:.4f}",
    token=HF_TOKEN
)
print(f"\nUpload complete! Artifacts published to:\nhttps://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")
